# Fine-tune MarkupLM for HTML phishing classification

One label per page: benign or phishing. This notebook selects the checkpoint with
the lowest validation loss, then restores it before testing, saving and inference.
Sampling covers multiple PhreshPhish shards, includes short HTML, and reserves
training-pool slots for short pages and labeled-benign login pages when available.

Restart the kernel and run all cells. A new cache and artifact directory keep this
experiment separate from the previous run. Initial sampling downloads several
parquet shards; later runs reuse the versioned sample cache. Full training needs PyTorch >=2.6.


In [ ]:
import json
import random
import hashlib
import math
import re
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlsplit

import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from bs4 import BeautifulSoup, Comment

import torch
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import MarkupLMFeatureExtractor, MarkupLMForSequenceClassification, MarkupLMProcessor, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = "microsoft/markuplm-base"
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
NUM_EPOCHS = 3  # maximum; best validation-loss weights are restored
EARLY_STOPPING_PATIENCE = 2
training_complete = False

# Context window. The base checkpoint ships 512; section 5 widens the position
# table to match this value. Measured on the PhreshPhish test split, 58% of pages
# overflow 512 tokens, and the overflow is lopsided — benign pages have a median
# of 1582 tokens against 392 for phishing ones. Truncating at 512 therefore fed
# the model whole phishing pages but only the navigation bar of real sites, and a
# length-only baseline scored the same AUC (0.749) as the trained v0 artifact.
MAX_TOKENS = 2048

# Attention is quadratic, so 2048 tokens will not fit four pages at once: a
# measured training step on 24 GB of unified memory needs ~4.8 GB at batch 1 and
# runs out of memory at batch 4. Micro-batches of 1 are accumulated instead, which
# keeps the effective batch at 4 and leaves the learning-rate schedule comparable
# to the 512-token run.
GRAD_ACCUM_STEPS = 4

ARTIFACT_DIR = Path("artifacts/phishing-html-classifier-v2")

id2label = {0: "benign", 1: "phishing"}
label2id = {v: k for k, v in id2label.items()}
PHRESHPHISH_LABEL_TO_ID = {"benign": 0, "phish": 1}


## 1. Diverse sampling and separate host groups

Sample uniformly within each label across spaced-apart shards, with extra training
coverage for short pages and benign login forms. Labels always come from the dataset;
the login heuristic only selects examples, never assigns labels. Test sampling has
no short/login quota. It remains class-balanced, so precision is not a browsing-traffic estimate.

Deduplicate full HTML by hash; split training/validation by normalized hostname and
exclude overlapping test hosts/content. This is hostname isolation, not a guarantee
that related domains or templates are independent. Insufficient short benign coverage
is reported explicitly; changing a filter cannot manufacture examples absent from the data.


In [ ]:
DATASET_ID = "phreshphish/phreshphish"
CACHE_DIR = Path("datasets/phreshphish_sample/html_v1_short_pages")
USE_TRAIN_PER_CLASS = 400
USE_VAL_PER_CLASS = 100
USE_TEST_PER_CLASS = 100
TRAIN_SHARDS = 4
TEST_SHARDS = 3
MIN_HTML_CHARS = 20
MAX_HTML_CHARS = 1_000_000
PARSE_CHAR_BUDGET = 300_000
SHORT_HTML_CHARS = 2_000
SHORT_FRACTION = 0.20
BENIGN_LOGIN_FRACTION = 0.20
LOGIN_RE = re.compile(r"<input\b[^>]*\btype\s*=\s*['\"]?password\b", re.I)


def normalize_page(row):
    html = row.get("html") or ""
    label = row.get("label")
    if label not in PHRESHPHISH_LABEL_TO_ID or not MIN_HTML_CHARS <= len(html) <= MAX_HTML_CHARS:
        return None
    url = row.get("url") or ""
    try:
        host = (urlsplit(url).hostname or "").lower().rstrip(".")
    except ValueError:
        host = ""
    # Hash before truncation, so different tails are not mistaken for identical pages.
    digest = hashlib.sha256(html.encode("utf-8", "ignore")).hexdigest()
    return {"html": html[:PARSE_CHAR_BUDGET], "html_len": len(html),
            "label": label, "label_id": PHRESHPHISH_LABEL_TO_ID[label], "url": url,
            "host": host, "sha256": digest,
            "is_login": bool(LOGIN_RE.search(html)), "short": len(html) <= SHORT_HTML_CHARS}


def host_group(row):
    return "host:" + row["host"] if row["host"] else "hash:" + row["sha256"]


def reservoir_add(bucket, row, count, capacity, rng):
    if len(bucket) < capacity:
        bucket.append(row)
    else:
        position = rng.randrange(count)
        if position < capacity:
            bucket[position] = row


def select_coverage(uniform, short, login, limit, short_fraction, login_fraction):
    selected, seen = [], set()
    def add(rows, target):
        added = 0
        for row in rows:
            if len(selected) >= limit or added >= target:
                break
            if row["sha256"] not in seen:
                selected.append(row)
                seen.add(row["sha256"])
                added += 1
    add(short, round(limit * short_fraction))
    add(login, round(limit * login_fraction))
    add(uniform, limit)
    return selected


def sample_shard(local_path, per_class, seed, enrich):
    rng = random.Random(seed)
    pools = {(label, kind): [] for label in (0, 1) for kind in ("uniform", "short", "login")}
    counts = Counter()
    for batch in pq.ParquetFile(local_path).iter_batches(batch_size=64, columns=["html", "url", "label"]):
        for raw in batch.to_pylist():
            row = normalize_page(raw)
            if row is None:
                continue
            label = row["label_id"]
            kinds = ["uniform"]
            if enrich and row["short"]:
                kinds.append("short")
            if enrich and label == 0 and row["is_login"]:
                kinds.append("login")
            for kind in kinds:
                key = (label, kind)
                counts[key] += 1
                reservoir_add(pools[key], row, counts[key], per_class, rng)
    selected = []
    for label in (0, 1):
        for kind in ("uniform", "short", "login"):
            rng.shuffle(pools[(label, kind)])
        selected += select_coverage(
            pools[(label, "uniform")], pools[(label, "short")], pools[(label, "login")],
            per_class, SHORT_FRACTION if enrich else 0,
            BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0,
        )
    return selected


def sample_split(split, per_class, shard_count, enrich):
    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    if not files:
        raise RuntimeError(f"No parquet shards for {split}")
    count = min(shard_count, len(files))
    indices = [round(i * (len(files) - 1) / max(count - 1, 1)) for i in range(count)]
    rows = []
    for index in indices:
        shard = files[index]
        local = hf_hub_download(DATASET_ID, shard, repo_type="dataset")
        sampled = sample_shard(local, math.ceil(per_class / count), SEED + index, enrich)
        for row in sampled:
            row["shard"] = shard
        rows.extend(sampled)
        print(shard, "selected:", len(sampled))
    return rows


def deduplicate(rows):
    by_hash, conflicting = {}, set()
    for row in rows:
        key = row["sha256"]
        if key in by_hash and by_hash[key]["label_id"] != row["label_id"]:
            conflicting.add(key)
        by_hash[key] = row
    return [r for key, r in by_hash.items() if key not in conflicting]


def choose_balanced(rows, per_class, seed, enrich=False):
    rng = random.Random(seed)
    result = []
    for label in (0, 1):
        group = [r for r in rows if r["label_id"] == label]
        rng.shuffle(group)
        result += select_coverage(group, [r for r in group if r["short"]],
                                  [r for r in group if r["is_login"] and label == 0], per_class,
                                  SHORT_FRACTION if enrich else 0,
                                  BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0)
    rng.shuffle(result)
    return result


def split_hosts(rows, val_per_class, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[host_group(row)].append(row)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    val, val_groups, counts = [], set(), Counter()
    for key in keys:
        group = groups[key]
        additions = Counter(r["label_id"] for r in group)
        if all(counts[label] + additions[label] <= val_per_class for label in (0, 1)):
            val.extend(group)
            counts.update(additions)
            val_groups.add(key)
    return [r for r in rows if host_group(r) not in val_groups], val


def assert_separate(splits):
    previous_hosts, previous_hashes = set(), set()
    for name, rows in splits.items():
        hosts, hashes = {host_group(r) for r in rows}, {r["sha256"] for r in rows}
        if hosts & previous_hosts or hashes & previous_hashes:
            raise ValueError(f"Host/content overlap in {name}")
        if len(hashes) != len(rows):
            raise ValueError(f"Duplicate content in {name}")
        if set(r["label_id"] for r in rows) != {0, 1}:
            raise ValueError(f"{name} needs both classes; increase shard/sample counts")
        previous_hosts.update(hosts)
        previous_hashes.update(hashes)


sample_config = {"version": 1, "dataset": DATASET_ID, "seed": SEED,
                 "train_shards": TRAIN_SHARDS, "test_shards": TEST_SHARDS,
                 "train_pool_per_class": 2 * (USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS),
                 "test_pool_per_class": 3 * USE_TEST_PER_CLASS,
                 "min_chars": MIN_HTML_CHARS, "max_chars": MAX_HTML_CHARS,
                 "parse_budget": PARSE_CHAR_BUDGET, "short_chars": SHORT_HTML_CHARS,
                 "short_fraction": SHORT_FRACTION, "login_fraction": BENIGN_LOGIN_FRACTION}
cache_metadata = CACHE_DIR / "sampling.json"
cache_valid = (cache_metadata.exists()
               and json.loads(cache_metadata.read_text(encoding="utf-8")) == sample_config
               and all((CACHE_DIR / f"{name}.jsonl").exists() for name in ("pool", "test")))
if cache_valid:
    def read_cache(name):
        with (CACHE_DIR / f"{name}.jsonl").open(encoding="utf-8") as handle:
            return [json.loads(line) for line in handle]
    pool, test_pool = read_cache("pool"), read_cache("test")
else:
    pool = sample_split("train", sample_config["train_pool_per_class"], TRAIN_SHARDS, True)
    test_pool = sample_split("test", sample_config["test_pool_per_class"], TEST_SHARDS, False)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in (("pool", pool), ("test", test_pool)):
        with (CACHE_DIR / f"{name}.jsonl").open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    cache_metadata.write_text(json.dumps(sample_config, indent=2), encoding="utf-8")

pool = deduplicate(pool)
train_pool, val_raw = split_hosts(pool, USE_VAL_PER_CLASS, SEED)
train_raw = choose_balanced(train_pool, USE_TRAIN_PER_CLASS, SEED, enrich=True)
known_hosts = {host_group(r) for r in train_raw + val_raw}
known_hashes = {r["sha256"] for r in pool}
test_pool = deduplicate(test_pool)
eligible_test = [r for r in test_pool if host_group(r) not in known_hosts and r["sha256"] not in known_hashes]
print("Excluded test pages with seen hosts/content:", len(test_pool) - len(eligible_test))
test_raw = choose_balanced(eligible_test, USE_TEST_PER_CLASS, SEED + 1)
assert_separate({"train": train_raw, "val": val_raw, "test": test_raw})
for name, rows in (("train", train_raw), ("val", val_raw), ("test", test_raw)):
    print(name, "labels:", dict(Counter(r["label_id"] for r in rows)),
          "short benign:", sum(r["short"] and r["label_id"] == 0 for r in rows),
          "benign login:", sum(r["is_login"] and r["label_id"] == 0 for r in rows))


## 2. Shared HTML preprocessing

Training and inference use the same HTML limit and parser. Remove scripts, styles,
templates and comments so boilerplate code does not consume the `MAX_TOKENS` window.
Failed or empty pages are recorded separately. Inspect the coverage table after parsing;
short raw HTML alone does not guarantee diverse usable text.

Scripts are still discarded here. Obfuscated JavaScript is one of the stronger
phishing tells, so this is a real blind spot, but changing it would alter what the
window is measured against — keep it fixed while testing the wider context.


In [ ]:
feature_extractor = MarkupLMFeatureExtractor()


def extract_page(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(["script", "style", "noscript", "template"]):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    encoding = feature_extractor(str(soup))
    pairs = [(node.strip(), xpath) for node, xpath in zip(encoding["nodes"][0], encoding["xpaths"][0])
             if node.strip()]
    if not pairs:
        raise ValueError("No usable text nodes")
    nodes, xpaths = zip(*pairs)
    return {"nodes": [list(nodes)], "xpaths": [list(xpaths)]}


parse_failures = []
def build_examples(raw_examples, split):
    examples = []
    for item in tqdm(raw_examples, desc=f"parse {split}"):
        try:
            parsed = extract_page(item["html"])
        except Exception as exc:
            parse_failures.append({"split": split, "url": item["url"], "sha256": item["sha256"],
                                   "label_id": item["label_id"], "reason": type(exc).__name__})
            continue
        examples.append({**{k: v for k, v in item.items() if k != "html"}, **parsed})
    return examples


train_data = build_examples(train_raw, "train")
val_data = build_examples(val_raw, "val")
test_data = build_examples(test_raw, "test")
assert_separate({"train": train_data, "val": val_data, "test": test_data})
coverage = {}
for name, data in (("train", train_data), ("val", val_data), ("test", test_data)):
    coverage[name] = {"n": len(data), "labels": dict(Counter(r["label_id"] for r in data)),
                      "short_benign": sum(r["short"] and r["label_id"] == 0 for r in data),
                      "short_phishing": sum(r["short"] and r["label_id"] == 1 for r in data),
                      "benign_login": sum(r["is_login"] and r["label_id"] == 0 for r in data),
                      "tiny_benign_under_200": sum(r["html_len"] < 200 and r["label_id"] == 0 for r in data)}
    print(name, coverage[name])
for group in ("short_benign", "benign_login", "tiny_benign_under_200"):
    if coverage["train"][group] < 10:
        print(f"COVERAGE GAP: only {coverage['train'][group]} training examples for {group}; "
              "more shards or additional independently labeled data are needed.")
print("Parse failures:", dict(Counter(r["split"] for r in parse_failures)))


## 3. PyTorch Dataset (one label per page, not per node)

In [ ]:
def encode_page(item, page_processor, max_length=MAX_TOKENS):
    return page_processor(nodes=item["nodes"], xpaths=item["xpaths"], truncation=True,
                          padding="max_length", max_length=max_length, return_tensors="pt")


class HtmlPhishingDataset(Dataset):
    """One HTML page, one integer label."""
    def __init__(self, data, processor, max_length=MAX_TOKENS):
        self.data, self.processor, self.max_length = data, processor, max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = encode_page(item, self.processor, self.max_length)
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = torch.tensor(item["label_id"], dtype=torch.long)
        return encoding


## 4. Processor, datasets, dataloaders

In [ ]:
processor = MarkupLMProcessor.from_pretrained("microsoft/markuplm-base")
processor.parse_html = False
# An explicit max_length already overrides this, but the value is persisted to the
# artifact's tokenizer_config.json. Left at 512 it would tell every downstream
# loader that the saved model is narrower than it really is.
processor.tokenizer.model_max_length = MAX_TOKENS

train_dataset = HtmlPhishingDataset(train_data, processor)
val_dataset = HtmlPhishingDataset(val_data, processor)
test_dataset = HtmlPhishingDataset(test_data, processor)

BATCH_SIZE = 1  # see GRAD_ACCUM_STEPS; batch 4 at 2048 tokens exhausts 24 GB
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

token_counts = [int(train_dataset[i]["attention_mask"].sum()) for i in range(min(64, len(train_dataset)))]
print(f"sampled {len(token_counts)} training pages: "
      f"median {sorted(token_counts)[len(token_counts) // 2]} tokens, "
      f"{sum(c >= MAX_TOKENS for c in token_counts)} still hitting the {MAX_TOKENS}-token cap")


## 5. Fresh model initialization and a wider context window

The training cell reloads the original checkpoint for every run, using float32.
PyTorch >= 2.6 is required to load the original `.bin` weights. The new classification
head is expected to be randomly initialized. Restart the kernel and run all cells
after changing dependencies; never resume from the previous NaN artifact.

`extend_position_embeddings` then widens the learned position table from 514 to
`MAX_TOKENS + 2`. Nothing else about the model changes — same 12 layers, same 12
heads, same attention. Only the lookup table that caps input length grows, at a
cost of about 1.2M parameters.

The copied positions start out as duplicates of the original 512, so the wider
window means nothing until fine-tuning pulls them apart. Attention also stays
quadratic: on 24 GB of unified memory a 2048-token training step measured ~3.4s
at batch 1 and ran out of memory at batch 4, which is why section 4 drops the
batch to 1 and accumulates gradients instead.


In [ ]:
def extend_position_embeddings(model, max_tokens):
    """Widen MarkupLM's learned position table without touching the architecture.

    MarkupLM uses absolute position embeddings: a lookup table of shape
    [514, 768], where 514 is 512 positions plus RoBERTa's two offset slots. Only
    that table caps the input length, so copying the learned block over and over
    until it reaches the new size buys a longer window for 1.2M extra parameters
    and no change to the layers, heads or attention. This is how the Longformer
    authors stretched RoBERTa from 512 to 4096.

    The tiling makes position 600 start out identical to position 88, so the
    widened positions carry no distinct meaning until training separates them.
    Fine-tuning is what makes the extra window real; loading this model and
    running inference without training would be no better than the 512 version.
    """
    target = max_tokens + 2  # keep RoBERTa's padding_idx offset
    embeddings = model.markuplm.embeddings
    old = embeddings.position_embeddings
    if old.num_embeddings >= target:
        return model

    widened = nn.Embedding(target, old.embedding_dim, padding_idx=old.padding_idx)
    with torch.no_grad():
        widened.weight[:2] = old.weight[:2]  # pad and offset slots are not positions
        learned = old.weight[2:]
        cursor = 2
        while cursor < target:
            span = min(learned.size(0), target - cursor)
            widened.weight[cursor:cursor + span] = learned[:span]
            cursor += span
    embeddings.position_embeddings = widened.to(old.weight.device, old.weight.dtype)
    # position_ids is a non-persistent buffer, so it is rebuilt from the config on
    # load; updating the config is what makes save/reload round-trip correctly.
    embeddings.register_buffer(
        "position_ids", torch.arange(target, device=old.weight.device).expand((1, -1)), persistent=False
    )
    model.config.max_position_embeddings = target
    return model


def create_model(device):
    torch.manual_seed(SEED)
    model = MarkupLMForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        use_safetensors=False,
        dtype=torch.float32,
    )
    before = model.markuplm.embeddings.position_embeddings.num_embeddings
    extend_position_embeddings(model, MAX_TOKENS)
    after = model.markuplm.embeddings.position_embeddings.num_embeddings
    print(f"position table {before} -> {after} ({(after - before) * model.config.hidden_size:,} new parameters)")
    model = model.to(device)
    assert_finite_parameters(model)
    return model


## 6. Metrics helper

In [ ]:
def require_finite(tensor, context):
    if not torch.isfinite(tensor).all().item():
        raise RuntimeError(f"Non-finite {context}. Stop and restart training from the base checkpoint.")


def assert_finite_parameters(model):
    for name, parameter in model.named_parameters():
        require_finite(parameter, f"parameter {name}")


@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for step, batch in enumerate(dataloader, start=1):
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        require_finite(outputs.logits, f"evaluation logits at batch {step}")
        require_finite(outputs.loss, f"evaluation loss at batch {step}")
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())
        total_loss += outputs.loss.item() * batch["labels"].numel()
    if not all_labels:
        raise ValueError("Cannot evaluate an empty dataset.")
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", pos_label=1, zero_division=0
    )
    return {
        "loss": total_loss / len(all_labels),
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision_phishing": precision,
        "recall_phishing": recall,
        "f1_phishing": f1,
        "prediction_counts": {id2label[i]: all_preds.count(i) for i in id2label},
        "n": len(all_labels),
    }, all_preds, all_labels


class ValidationLossTracker:
    """Save every strictly better loss; stop after patience consecutive non-improvements."""
    def __init__(self, patience):
        if patience < 1:
            raise ValueError("Patience must be positive")
        self.patience = patience
        self.best_loss, self.best_epoch, self.stale = float("inf"), None, 0

    def update(self, epoch, loss):
        if not math.isfinite(loss):
            raise ValueError("Non-finite validation loss")
        improved = loss < self.best_loss
        if improved:
            self.best_loss, self.best_epoch, self.stale = loss, epoch, 0
        else:
            self.stale += 1
        return improved, self.stale >= self.patience


## 7. Train and restore the best validation-loss checkpoint

`NUM_EPOCHS` is a maximum, not the epoch that will be exported. Stop after two
consecutive epochs without improvement, then restore the lowest validation-loss
weights before testing or saving. No test or smoke-test outcome selects the checkpoint.
Loss selection limits overconfidence but does not guarantee a specific example passes.


In [ ]:
training_complete = False
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

for name, data in (("train", train_data), ("val", val_data), ("test", test_data)):
    counts = Counter(item["label_id"] for item in data)
    if set(counts) != {0, 1}:
        raise ValueError(f"{name} must contain both classes; got {counts}")
    print(f"{name} label counts: {dict(counts)}")

# Release an earlier notebook model before allocating another one on the GPU.
if "model" in globals():
    del model
if "optimizer" in globals():
    del optimizer
if "scheduler" in globals():
    del scheduler
model = create_model(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
# The scheduler advances once per optimizer step, not once per micro-batch.
steps_per_epoch = math.ceil(len(train_dataloader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)
print(f"{len(train_dataloader)} micro-batches/epoch -> {steps_per_epoch} optimizer steps "
      f"(effective batch {BATCH_SIZE * GRAD_ACCUM_STEPS})")
history = []
tracker = ValidationLossTracker(EARLY_STOPPING_PATIENCE)
best_state = None

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, seen = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(train_dataloader, desc=f"epoch {epoch + 1}/{NUM_EPOCHS}")
    for step, batch in enumerate(progress, start=1):
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        context = f"epoch {epoch + 1}, batch {step}"
        require_finite(outputs.logits, f"training logits ({context})")
        loss = outputs.loss
        require_finite(loss, f"training loss ({context})")
        # Scale so accumulated gradients average rather than sum.
        (loss / GRAD_ACCUM_STEPS).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_dataloader):
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        labels = inputs["labels"]
        seen += labels.numel()
        running_loss += loss.item() * labels.numel()
        correct += (outputs.logits.detach().argmax(dim=-1) == labels).sum().item()
        progress.set_postfix(loss=running_loss / seen, accuracy=correct / seen)

    assert_finite_parameters(model)
    val_metrics, _, _ = evaluate_model(model, val_dataloader, device)
    entry = {
        "epoch": epoch + 1,
        "train_loss": running_loss / seen,
        "train_accuracy": correct / seen,
        "val": val_metrics,
    }
    history.append(entry)
    print(entry)
    improved, should_stop = tracker.update(epoch + 1, val_metrics["loss"])
    if improved:
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        print(f"Best checkpoint: epoch {tracker.best_epoch}, val_loss={tracker.best_loss:.4f}")
    if should_stop:
        print("Early stopping: validation loss stopped improving.")
        break

if best_state is None:
    raise RuntimeError("No valid checkpoint was produced")
model.load_state_dict(best_state)
del best_state
assert_finite_parameters(model)
model.eval()
best_epoch = tracker.best_epoch
restored_val_metrics, _, _ = evaluate_model(model, val_dataloader, device)
print(f"Restored epoch {best_epoch}:", restored_val_metrics)
training_complete = True


## 8. Evaluate on the held-out test sample

In [ ]:
if not training_complete:
    raise RuntimeError("Complete a finite training run before testing.")

test_metrics, test_preds, test_labels = evaluate_model(model, test_dataloader, device)
print("test:", test_metrics)
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(test_labels, test_preds, labels=[0, 1]))

slice_metrics = {}
for name, subset in (
    ("short_pages", [r for r in test_data if r["short"]]),
    ("short_benign", [r for r in test_data if r["short"] and r["label_id"] == 0]),
    ("benign_login", [r for r in test_data if r["is_login"] and r["label_id"] == 0]),
    ("longer_pages", [r for r in test_data if not r["short"]]),
):
    if not subset:
        print(name, "unavailable: no examples in this test sample")
        continue
    loader = DataLoader(HtmlPhishingDataset(subset, processor), batch_size=BATCH_SIZE)
    scores, predictions, labels = evaluate_model(model, loader, device)
    negatives = sum(label == 0 for label in labels)
    scores["false_positive_rate"] = (sum(p == 1 and y == 0 for p, y in zip(predictions, labels)) / negatives
                                     if negatives else None)
    scores["label_counts"] = dict(Counter(labels))
    slice_metrics[name] = scores
    print(name, scores)
print("Small slices are diagnostics; report their sample counts with their metrics.")


## 9. Save the fine-tuned artifact

In [ ]:
if not training_complete:
    raise RuntimeError("Refusing to save: training did not complete successfully.")
assert_finite_parameters(model)
# Re-evaluate the current model rather than trusting stale notebook metrics.
test_metrics, test_preds, test_labels = evaluate_model(model, test_dataloader, device)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ARTIFACT_DIR, safe_serialization=True)
processor.save_pretrained(ARTIFACT_DIR)

metadata = {
    "base_model": BASE_MODEL,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_grad_norm": MAX_GRAD_NORM,
    "seed": SEED,
    "history": history,
    "best_epoch": best_epoch,
    "selection_metric": "lowest validation cross-entropy",
    "restored_validation": restored_val_metrics,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "max_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
    "sampling": sample_config,
    "coverage": coverage,
    "slice_metrics": slice_metrics,
    "preprocessing": {"parse_char_budget": PARSE_CHAR_BUDGET,
                      "strip_tags": ["script", "style", "noscript", "template"],
                      "strip_comments": True, "max_tokens": MAX_TOKENS},
    "context_window": {
        "max_tokens": MAX_TOKENS,
        "base_max_tokens": 512,
        "position_embeddings": model.config.max_position_embeddings,
        "method": "tiled the base checkpoint's 512 learned position embeddings to fill the wider table",
        "note": "Inference must use this max_tokens; loading these weights with a 512-token window "
                "discards most of what the model was trained to read.",
    },
    "task": "sequence-classification",
    "id2label": id2label,
    "train_dataset": "phreshphish/phreshphish",
    "train_n": len(train_data),
    "val_n": len(val_data),
    "test_n": len(test_data),
    "test_metrics": test_metrics,
    "baselines": {
        "length_only_auc": 0.749,
        "markuplm_v0_auc": 0.751,
        "note": "Measured on 800 held-out PhreshPhish pages. Counting tokens alone matched the "
                "512-token v0 artifact, so this run has to beat 0.749 to show it learned anything "
                "beyond document length.",
    },
    "note": "Small demo sample — not production-grade. Scale up USE_*_PER_CLASS and retrain before relying on this.",
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, allow_nan=False))
print("saved to", ARTIFACT_DIR.resolve())
for filename, rows in (
    ("parse_failures.jsonl", parse_failures),
    ("split_manifest.jsonl", [{"split": name, **{k: r[k] for k in ("url", "host", "sha256", "label_id", "html_len")}}
                               for name, data in (("train", train_raw), ("val", val_raw), ("test", test_raw))
                               for r in data]),
):
    with (ARTIFACT_DIR / filename).open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


## 10. Reload parity check and independent smoke examples

First compare original and reloaded logits on real test pages. Then inspect a small
handwritten suite that is never added to training. These are diagnostic examples,
not a representative accuracy benchmark. Scores are raw softmax, not calibrated
probabilities of risk. A confidently wrong example remains a failure to investigate.


In [ ]:
loaded_processor = MarkupLMProcessor.from_pretrained(ARTIFACT_DIR)
loaded_processor.parse_html = False
loaded_model = MarkupLMForSequenceClassification.from_pretrained(ARTIFACT_DIR).to(device)
loaded_model.eval()

# A save/reload mismatch is an implementation issue, separate from model quality.
model.eval()
parity_batch = next(iter(test_dataloader))
parity_inputs = {k: v.to(device) for k, v in parity_batch.items() if k != "labels"}
with torch.no_grad():
    before = model(**parity_inputs).logits
    after = loaded_model(**parity_inputs).logits
torch.testing.assert_close(before, after, rtol=1e-4, atol=1e-5)
print("PASS: original and reloaded checkpoint agree on a test batch.")

samples = {
    "plain_benign": "<html><head><title>My Blog</title></head><body><h1>Welcome</h1><p>Just a personal blog about hiking.</p></body></html>",
    "recipe_benign": "<html><title>Oatmeal</title><body><h1>Breakfast recipe</h1><p>Simmer oats in water for five minutes. Serve with fruit.</p></body></html>",
    "contact_benign": "<html><title>Community Library</title><body><h1>Opening hours</h1><p>Tuesday through Saturday, 9am to 5pm.</p></body></html>",
    "login_benign": "<html><title>Sign in - Example Library</title><body><h1>Member sign in</h1><form action='/login'><label>Email</label><input type='email'><label>Password</label><input type='password'><button>Sign in</button></form><a href='/help'>Contact library staff</a></body></html>",
    "fake_login": """
    <html><head><title>Verify Your Account - Secure Login</title></head>
    <body>
    <img src=\"logo.png\" alt=\"PayPal\">
    <h2>Your account has been limited. Please verify your identity.</h2>
    <form action=\"http://totally-not-paypal.example.ru/steal\" method=\"post\">
    <input type=\"text\" name=\"email\" placeholder=\"Email\">
    <input type=\"password\" name=\"password\" placeholder=\"Password\">
    <input type=\"text\" name=\"card_number\" placeholder=\"Card number\">
    <button type=\"submit\">Confirm Identity</button>
    </form>
    </body></html>
    """,
}


smoke_results = []
for name, html in samples.items():
    parsed = extract_page(html)
    inputs = encode_page(parsed, loaded_processor).to(device)
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
    require_finite(logits, f"inference logits for {name}")
    prob = torch.softmax(logits, dim=-1)[0, 1].item()
    predicted = int(prob >= 0.5)
    expected = 1 if name == "fake_login" else 0
    result = {"name": name, "expected": id2label[expected], "predicted": id2label[predicted],
              "phishing_score": prob, "correct": predicted == expected,
              "html_chars": len(html), "input_tokens": inputs["attention_mask"].sum().item()}
    smoke_results.append(result)
    print(result)
(ARTIFACT_DIR / "smoke_results.json").write_text(json.dumps(smoke_results, indent=2, allow_nan=False))
print("Do not tune training on this tiny suite; use independent short-page test coverage.")
